In [13]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'automa.svg'
root_svg = apri_file_svg(nome_file_svg)

# Trova l'elemento con id="automa1"
for child in root_svg:
    if child.attrib['id'] == 'layer1':
        automa1 = child

stati = []
finalStates = []
transizioni = []
transizioni_linguaggio = []
initialState = None

for child in automa1:
    if not re.search('title', child.attrib['id']):
        elemento = child.attrib['id']

        # stato iniziale
        if re.search('start', elemento):
            elemento = elemento.replace('start-', '')
            initialState = elemento

        # stati e stati finali
        elif re.search('stato', elemento):
            stato = elemento.replace('stato-', '')

            if re.search('finale', stato):
                stato = stato.replace('-finale', '')
                finalStates.append(stato)

            stati.append(stato)

        # transizioni
        elif re.search('transizione', elemento):
            trans = elemento.replace('transizione-', '').split('-')
            sorgente, destinazione = trans[0], trans[1]
            simboli = []

            for child2 in child:
                if re.search('valore', child2.attrib['id']):
                    for child3 in child2:
                        simboli.append(child3.text)

            # se non ci sono simboli, la transizione ha valore None
            if simboli:
                for s in simboli:
                    transizioni.append((sorgente, destinazione, s))
            else:
                transizioni.append((sorgente, destinazione, None))

# ricostruzione struttura dati
automa = {
    "initialState": initialState,
    "finalStates": finalStates,
    "states": stati,
    "transitions": transizioni
}

print(automa)


{'initialState': 'q0', 'finalStates': ['q0'], 'states': ['q4', 'q3', 'q2', 'q1', 'q0'], 'transitions': [('q4', 'q0', '0'), ('q3', 'q4', '0'), ('q2', 'q3', '0'), ('q1', 'q2', '1'), ('q0', 'q1', '1')]}


In [14]:
import re

def filtra_automa(automa, query):
    result = {
        "initialState": None,
        "finalStates": [],
        "states": [],
        "transitions": []
    }

    for q in query:
        # stato iniziale
        if q.startswith("start-"):
            stato = q.replace("start-", "")
            if stato == automa["initialState"]:
                result["initialState"] = stato

        # stati finali
        elif q.startswith("stato-") and q.endswith("-finale"):
            stato = q.replace("stato-", "").replace("-finale", "")
            if stato in automa["finalStates"]:
                result["finalStates"].append(stato)

        # stati
        elif q.startswith("stato-") and not q.endswith("-finale"):
            stato = q.replace("stato-", "")
            if stato in automa["states"]:
                result["states"].append(stato)

        # transizione generica (simbolo = "?")
        elif q.startswith("transizione-"):
            sorg, dest = q.replace("transizione-", "").split("-")
            if any(t[0] == sorg and t[1] == dest for t in automa["transitions"]):
                result["transitions"].append([sorg, dest, "?"])

        # valore specifico (simbolo vero dall’automa)
        elif q.startswith("valore-"):
            sorg, dest = q.replace("valore-", "").split("-")
            for t in automa["transitions"]:
                if t[0] == sorg and t[1] == dest:
                    result["transitions"].append([sorg, dest, t[2]])

    # rimuovo duplicati mantenendo priorità dei valori rispetto a '?'
    clean_transitions = {}
    for s, d, val in result["transitions"]:
        key = (s, d)
        if key not in clean_transitions:
            clean_transitions[key] = set()
        clean_transitions[key].add(val)

    final_transitions = []
    for (s, d), vals in clean_transitions.items():
        if len(vals) > 1 and "?" in vals:
            vals.remove("?")  # elimina ? se c'è anche un valore
        for v in vals:
            final_transitions.append([s, d, v])  # <-- LISTA invece di tupla

    result["transitions"] = final_transitions

    # rimuovo duplicati in stati/finali
    result["finalStates"] = list(set(result["finalStates"]))
    result["states"] = list(set(result["states"]))

    return result


valori accettati nella query
- start-q* 				-> initialState: 'q*'
- transizione-q*-q* 	-> transitons: [(q*, q*, ?)]
- valore-q*-q* 			-> transitons: [(q*, q*, $('#valore-q*-q*'))]
- stato-q*-finale 		-> finalStates: ['q*']
- stato-q*				-> states: ['q*'] 


In [15]:
automa = {
    'initialState': 'q0',
    'finalStates': ['q0'],
    'states': ['q4', 'q3', 'q2', 'q1', 'q0'],
    'transitions': [('q4', 'q0', '0'), ('q3', 'q4', '0'), ('q2', 'q3', '0'), ('q1', 'q2', '1'), ('q0', 'q1', '1')]}


query = [
  "start-q0",
  "transizione-q1-q2",
  "valore-q1-q2",
  "valore-q2-q3",
  "stato-q0-finale",
  "stato-q1",
  "stato-q2"
]

filtrato = filtra_automa(automa, query)
print(filtrato)


{'initialState': 'q0', 'finalStates': ['q0'], 'states': ['q1', 'q2'], 'transitions': [['q1', 'q2', '1'], ['q2', 'q3', '0']]}
